In [ ]:
using Plots
using LinearAlgebra
plotly()

In [ ]:
# Nomalised wavelength for ideal geometry compariso
λ = 1.0

println("Normalised wavelength λ = ", λ)

In [ ]:
# Number of antenna elements
N = 3

# Angular position of each antenna around the UCA
alpha = (0:N-1) .* (2 * π / N)

# Degrees for easier inspection
alpha_deg = rad2deg.(alpha)

println("Antenna angles = ", alpha_deg)

In [ ]:
# Adjacent antenna spacing for the first comparison
d = 0.75 * λ

# UCA radius corresponding to adjacent chord spacing d
R = d / (2 * sin(π / N))

println("Adjacent spacing d/λ = ", d / λ)
println("Array radius R/λ = ", R / λ)

In [ ]:
# Plot the three-element UCA geometry
p = scatter(
    x ./ λ, y ./ λ,
    xlabel = "x / λ",
    ylabel = "y / λ",
    label = false,
    markersize = 8,
    aspect_ratio = :equal,
    title = "Three-Element Uniform Circular Array"
)

scatter!([0], [0],
    label = false,
    markersize = 5
)

for n = 1:N
    annotate!(x[n] / λ, y[n] / λ + 0.015, text("A$n", 10))
end

display(p)

In [ ]:
# True source azimuth
theta_true_deg = 30.0
theta_true = deg2rad(theta_true_deg)

# Path difference relative to reference antenna A1
delta_r = R .* (
    cos(theta_true) .-
    cos.(theta_true .- alpha)
)

for n = 1:N
    println(
        "Antenna ", n,
        ": Δr_1", n, "/λ = ", delta_r[n] / λ
    )
end

In [ ]:
# Reference-relative phase differences for the true source direction
delta_phi = (2 * π / λ) .* delta_r

# Wrap phases to [-π, π]
delta_phi_wrapped = angle.(exp.(im .* delta_phi))

# Convert to degrees for inspection
delta_phi_deg = rad2deg.(delta_phi_wrapped)

for n = 1:N
    println(
        "Antenna ", n,
        ": Δφ_1", n, " = ", delta_phi_deg[n], " degrees"
    )
end

In [ ]:
# Simulated measured phase vector
# A1 is the reference, so only A2...AN are included
phi_meas = delta_phi_deg[2:end]

println("Measured reference-relative phase vector:")
println(phi_meas)

In [ ]:
# Candidate azimuths
theta_scan_deg = 0.0:0.1:359.9

# RMSE and direct phase-match score
J = zeros(length(theta_scan_deg))
score = zeros(length(theta_scan_deg))

# Include A1 = 0° as the phase reference
phase_meas_full = vcat(0.0, deg2rad.(phi_meas))
a_meas = exp.(im .* phase_meas_full)

for k = eachindex(theta_scan_deg)

    theta = deg2rad(theta_scan_deg[k])

    # Expected path differences relative to A1
    delta_r_exp = R .* (
        cos(theta) .-
        cos.(theta .- alpha)
    )

    # Expected reference-relative phases
    delta_phi_exp_rad = angle.(
        exp.(im .* (2 * π / λ) .* delta_r_exp)
    )

    delta_phi_exp_deg = rad2deg.(delta_phi_exp_rad[2:end])

    # -----------------------------
    # Wrapped phase-error RMSE
    # -----------------------------
    error = rad2deg.(
        angle.(exp.(im .* deg2rad.(phi_meas .- delta_phi_exp_deg)))
    )

    J[k] = sqrt(sum(error .^ 2) / (N - 1))

    # -----------------------------
    # Direct phase-match score
    # -----------------------------
    a_exp = exp.(im .* delta_phi_exp_rad)

    score[k] = abs(sum(conj.(a_exp) .* a_meas)) / N
end

# Direction estimates from both methods
theta_est_rmse = theta_scan_deg[argmin(J)]
theta_est_score = theta_scan_deg[argmax(score)]

println("True direction        = ", theta_true_deg, " degrees")
println("RMSE estimate         = ", theta_est_rmse, " degrees")
println("Phase-match estimate  = ", theta_est_score, " degrees")
println("Maximum score         = ", maximum(score))

In [ ]:
# Perfect-match locations for plotting
tolerance = 1e-6
perfect_indices = findall(J .< tolerance)
perfect_angles = theta_scan_deg[perfect_indices]

p = plot(
    deg2rad.(theta_scan_deg),
    score,
    proj = :polar,
    linewidth = 2,
    label = false,
    title = "UCA Phase Match, d = $(round(d / λ, digits=2))λ",
    titlefontsize = 11,
    rlims = (0, 1.1),
    rticks = 0:0.25:1.0,
    thetagrids = 0:45:315,
    gridalpha = 0.65,
    gridlinewidth = 0.8,
    size = (760, 600)
)
# Mark and label the exact phase-match direction(s)
match_labels = [
    "($(round(Int, theta_scan_deg[i]))°, $(round(score[i], digits=2)))"
    for i in perfect_indices
]

scatter!(
    p,
    deg2rad.(perfect_angles),
    score[perfect_indices],
    markersize = 5,
    markerstrokewidth = 1,
    markerstrokecolor = :black,
    label = false,
    series_annotations = match_labels
)

display(p)

d_name = replace(string(round(d / λ, digits=2)), "." => "p")
savefig(p, "UCA_PhaseMatch_d_$(d_name)lambda.png")

In [ ]:
d_name = replace(string(round(d / λ, digits=2)), "." => "p")
savefig("UCA_PhaseMatch_d_$(d_name)lambda.png")

In [ ]:
tolerance = 1e-6

perfect_indices = findall(J .< tolerance)
perfect_angles = theta_scan_deg[perfect_indices]

println("Perfect-match directions = ", perfect_angles)

In [ ]:
score = score ./ maximum(score)

plot(deg2rad.(phi_test_deg), score,
    proj = :polar,
    label = false,
    linewidth = 2,
    title = "UCA Direction-of-Arrival Response")

In [ ]:
##
# Detection of the direction of the main lobe and the second strongest lobe 
##
main_index = argmax(score)

# Find local peaks, including across the 0°/360° boundary
is_peak = (score.>circshift(score, 1)) .& (score.>circshift(score, -1))

peak_indices = findall(is_peak)

# Remove the main peak and find the strongest remaining peak
side_indices = filter(i -> i  != main_index, peak_indices)
side_index = side_indices[argmax(score[side_indices])]

main_angle = phi_test_deg[main_index]
side_angle = phi_test_deg[side_index]

# Main peak relative to strongest incorrect peak
SLL_dB = 20 * log10(score[main_index] / score[side_index])

println("Main peak angle      = ", main_angle, " degrees")
println("Strongest other peak = ", side_angle, " degrees")
println("Sidelobe separation  = ", SLL_dB, " dB")

In [ ]:
half_power = 1 / sqrt(2)

# Shift the main peak to the middle to avoid the 0°/360° boundary
centre = div(length(score), 2) + 1
score_shifted = circshift(score, centre - main_index)

left = centre
while left > 1 && score_shifted[left] >= half_power
    left -= 1
end

right = centre
while right < length(score) && score_shifted[right] >= half_power
    right += 1
end

HPBW = (right - left - 2) * step(phi_test_deg)

println("HPBW = ", HPBW, " degrees")